# K-Nearest Neighbors Demo

This simple demo accompanies the readings and the lecture. 

* It shows you how to fit a model using KNN using Python/sklearn. 
* It shows you how to compare more than one model using a validation set.
* If we had more time we would choose a best model and then evaluate that on the test set.

In [81]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

## Import and prepare data

### Import the dataset

In [82]:
# import example dataset
iris = datasets.load_iris() 
X = iris.data
y = iris.target

### Train test split

In [83]:
# 80/20 train test split: stratify on class label y
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.20,
                                                    stratify=y, 
                                                    random_state=2022)

### Validation set

In this exercise we will explore the performance of three "models" (ML methods + hyperparameters):

* Logistic Regression
* KNN Classification with $k=5$ and Euclidean Distance
* KNN Classification with $k=5$ and Manhattan Distance

Of course, we could try other models, or even perform grid search. We will choose the best of the three models on a validation set. 

To obtain the validation set we will further partition the training set 75/25. So now we have three sets, training, validation, test, split 60/20/20. Note that sometimes we do this explicitly, but more often in practice the validation set is handled by automated tools such as K-fold cross validation. 

In [84]:
# Validation set: 75/25 tts on the training set
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train,
                                                      test_size=0.25,
                                                      stratify=y_train, 
                                                      random_state=2022)

### Scale data

This time we will use standard scaler. Remember to fit on training test and then scale test set on the learned parameters (here the mean and variance). Since we are manually working with a validation set, we will scale that separately too. 

In [85]:
# scale data to simulate N(0,1) for every explanatory variable
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # shortcut
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

### First Model: Logistic Regression 

We start with logistic regression to provide a baseline and also for review.

#### Initialize the classifier and fit the model

In [86]:
# baseline: linear regression model
model = LogisticRegression(solver='lbfgs', 
                           multi_class='multinomial', 
                           C=1.0,
                           random_state=0)
model.fit(X_train, y_train)

LogisticRegression(multi_class='multinomial', random_state=0)

In [87]:
model.coef_

array([[-0.9040916 ,  1.19164546, -1.54343499, -1.47313946],
       [ 0.51979854, -0.54917234, -0.42074143, -0.58837707],
       [ 0.38429306, -0.64247313,  1.96417642,  2.06151653]])

In [88]:
model.intercept_

array([-0.18618418,  1.61682571, -1.43064154])

#### Evaluate the model on the validation set

In [89]:
model.score(X_valid, y_valid)

0.9666666666666667

The model is about 97% accurate. Let's look more closely. Remember that logistic regression outputs probabilities as well as predictions. Here are predictions use the default 50/50 threshold (but we'd have to do this ourselves using `y_proba` since I don't think sklearn does this for us with `y_predict()`). 

In [90]:
y_proba = model.predict_proba(X_valid)
y_predict = model.predict(X_valid)

In [91]:
y_proba[:8,:].round(2)

array([[0.  , 0.18, 0.81],
       [0.  , 0.02, 0.98],
       [0.  , 0.59, 0.4 ],
       [0.02, 0.91, 0.07],
       [0.02, 0.87, 0.12],
       [0.  , 0.04, 0.96],
       [0.01, 0.29, 0.7 ],
       [0.  , 0.21, 0.79]])

Notice that there are three classes, so we get the probability of class membership in each. The rows should thus add to one (since we are using the `multinomial` method for multiclass classification, which is based on softmax. 

Let's also look at the confusion matrix, precision, recall, etc.

In [92]:
confusion_matrix(y_predict, y_valid)

array([[10,  0,  0],
       [ 0, 10,  1],
       [ 0,  0,  9]])

In [93]:
print(classification_report(y_valid, y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.91      1.00      0.95        10
           2       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



Bottom line: Our model mixed up one instance. Note that this is a very small sample, so we can't rely too much on this, but it does give us a way to quantify how well we did.

### k-Nearest Neighbors Classifier ($k=5$, Euclidean distance)

We now use KNN non-parametric method. Note that there is no random state for this one. 



### k-Nearest Neighbors Classifier ($k=3$, Manhattan distance)

Let's try the nearest 3 neighbors using Manhattan distance as the metric.



In [94]:
# baseline: linear regression model
model = KNeighborsClassifier(n_neighbors=3, metric='manhattan')
model.fit(X_train, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=3)

#### Evaluate the model on the validation set

In [95]:
model.score(X_valid, y_valid)

y_predict = model.predict(X_valid)
y_proba = model.predict_proba(X_valid)

In [96]:
y_proba[:8,:].round(2)

array([[0.  , 0.  , 1.  ],
       [0.  , 0.  , 1.  ],
       [0.  , 0.  , 1.  ],
       [0.  , 1.  , 0.  ],
       [0.  , 1.  , 0.  ],
       [0.  , 0.  , 1.  ],
       [0.  , 0.67, 0.33],
       [0.  , 0.  , 1.  ]])

Notice we can still get probabilities with KNN, but it is the "local neighborhood" probability that we saw in class, so it is not very smooth. 

In [97]:
confusion_matrix(y_predict, y_valid)

array([[10,  0,  0],
       [ 0,  9,  2],
       [ 0,  1,  8]])

In [98]:
print(classification_report(y_valid, y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.82      0.90      0.86        10
           2       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



## Choose best model and evaluate on test set


The KNN methods did pretty well, but logistic regression wins out on the validation set. It's also a little faster at predicting new data points&mdash;although on this small, simple dataset there is no real difference. So let's go with the linear model.

There is no requirement to do this, but I often retrain the model using all the training set, so let's stitch the training set and validation set back together and rescale. Again, as a reminder, don't touch the test set until the end. 

In [99]:
X_train = np.concatenate([X_train, X_valid], axis=0) # using NumPy for this; can also be done in Pandas
y_train = np.concatenate([y_train, y_valid], axis=0)

In [100]:
X_train = scaler.fit_transform(X_train)

In [103]:
# baseline: linear regression model
model = LogisticRegression(solver='lbfgs', 
                           multi_class='multinomial', 
                           C=1.0,
                           random_state=0)
model.fit(X_train, y_train)

LogisticRegression(multi_class='multinomial', random_state=0)

In [104]:
model.score(X_valid, y_valid)

y_predict = model.predict(X_valid)
y_proba = model.predict_proba(X_valid)

In [105]:
y_proba[:8,:].round(2)

array([[0.  , 0.15, 0.85],
       [0.  , 0.02, 0.98],
       [0.  , 0.57, 0.43],
       [0.02, 0.93, 0.05],
       [0.01, 0.9 , 0.09],
       [0.  , 0.03, 0.97],
       [0.  , 0.3 , 0.7 ],
       [0.  , 0.18, 0.82]])

In [106]:
print(classification_report(y_valid, y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [107]:
confusion_matrix(y_predict, y_valid)

array([[10,  0,  0],
       [ 0, 10,  0],
       [ 0,  0, 10]])

### Assessment

We got 100%! That's exciting, but again, that doesn't mean we have a perfect classifier. It also doesn't necessarily mean we made a mistake (even though you should always be a little suspicious of 100% accuracy!). Keep in mind:

* This is a simple (toy) problem.
* This is a very small sample size ($N=30$), so we would expect a lot of variance depending on how lucky or unlucky we are with train-test split. 

Still, these are good results. 

## Conclusion

In this demo we have introduced KNN classification as well as validating and choosing a best model. 

Note that I wrote this very quickly. If you discover an error or have questions, please contact me so that we can discuss it. -Prof. Allen